# Notebook 03 — Silver to Gold

**Project:** End-to-End Retail Data Engineering Pipeline (Olist)

## Objective
Build business-ready **Gold Delta tables** from the validated Silver tables: a small star schema (dimensions + two fact tables) and five analytics tables used by SQL and Power BI.

## Design rules
- **Grain first.** Every table has one clearly stated grain (one row per ...).
- **No row multiplication.** Items, payments and reviews are never joined to each other. Each is aggregated to *order* grain first and then joined one-to-one to orders. Analytics tables are built from the two fact tables, not from raw Silver joins.
- **Validate every table.** Section 13 checks row counts and totals against Silver with assertions.

## Business definitions
| Term | Definition |
|---|---|
| Revenue order | Order whose status is not `canceled` or `unavailable` |
| GMV | Sum of item `price` for revenue orders (freight is reported separately) |
| Delivered order | Status `delivered` **and** a delivery date present (8 source orders are marked delivered without a date and are treated as not delivered) |
| Late order | Delivered on a date after the estimated delivery date |
| Order review score | Score of the **latest** review of the order (`review_count` is also stored) |
| Product category | English name, else Portuguese name, else `unknown` |

Source: `silver_*` tables. Destination: `gold_*` tables. Run Notebooks 01 and 02 first.


## 1. Configuration

Load the Silver tables once and define one helper that saves a Gold table (overwrite) and prints its row count. Gold is rebuilt in full on every run.

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

orders   = spark.table("silver_orders")
items    = spark.table("silver_order_items")
payments = spark.table("silver_order_payments")
reviews  = spark.table("silver_order_reviews")
cust     = spark.table("silver_customers")
products = spark.table("silver_products")
sellers  = spark.table("silver_sellers")
cat_map  = spark.table("silver_category_translation")


def save_gold(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"gold_{name}")
    print(f"gold_{name}: {spark.table(f'gold_{name}').count():,} rows")

StatementMeta(, d637f757-c75f-4c7b-ae00-ee5bde2d0359, 3, Finished, Available, Finished, False)

## 2. Dimension: Date

**Grain:** one row per calendar day, from the first to the last order purchase date. `date_key` (`yyyyMMdd`) joins to the facts.

In [2]:
first_day, last_day = orders.agg(F.min(F.to_date("order_purchase_timestamp")), F.max(F.to_date("order_purchase_timestamp"))).first()

dim_date = (
    spark.sql(f"SELECT explode(sequence(to_date('{first_day}'), to_date('{last_day}'), interval 1 day)) AS calendar_date")
    .select(
        F.date_format("calendar_date", "yyyyMMdd").cast("int").alias("date_key"),
        "calendar_date",
        F.year("calendar_date").alias("year"),
        F.quarter("calendar_date").alias("quarter"),
        F.month("calendar_date").alias("month"),
        F.date_format("calendar_date", "MMM").alias("month_name"),
        F.date_format("calendar_date", "yyyy-MM").alias("year_month"),
        F.date_format("calendar_date", "EEE").alias("day_name"),
        F.dayofweek("calendar_date").isin(1, 7).alias("is_weekend"),
    )
)
save_gold(dim_date, "dim_date")

StatementMeta(, d637f757-c75f-4c7b-ae00-ee5bde2d0359, 4, Finished, Available, Finished, False)

gold_dim_date: 774 rows


## 3. Dimension: Customer

**Grain:** one row per real customer (`customer_unique_id`). The source `customer_id` changes with every order, so it cannot identify a person. Location is taken from the customer's **most recent order**.

Expected: 96,096 rows.

In [3]:
latest_order = Window.partitionBy("customer_unique_id").orderBy(
    F.col("order_purchase_timestamp").desc(), F.col("order_id").desc())

dim_customer = (
    orders.join(cust, "customer_id")
    .withColumn("_rn", F.row_number().over(latest_order)).filter("_rn = 1")
    .select("customer_unique_id",
            F.col("customer_zip_code_prefix").alias("zip_prefix"),
            F.col("customer_city").alias("city"),
            F.col("customer_state").alias("state"))
)
save_gold(dim_customer, "dim_customer")

StatementMeta(, d637f757-c75f-4c7b-ae00-ee5bde2d0359, 5, Finished, Available, Finished, False)

gold_dim_customer: 96,096 rows


## 4. Dimension: Product

**Grain:** one row per product. The category falls back from English to Portuguese to `unknown`, so the 2 untranslated categories and the products without a category stay visible instead of disappearing.

Expected: 32,951 rows.

In [4]:
dim_product = (
    products.join(cat_map, "product_category_name", "left")
    .select(
        "product_id",
        F.coalesce("product_category_name_english", "product_category_name", F.lit("unknown")).alias("category"),
        F.col("product_category_name").alias("category_pt"),
        "product_name_length", "product_description_length", "product_photos_qty",
        "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
    )
)
save_gold(dim_product, "dim_product")

StatementMeta(, d637f757-c75f-4c7b-ae00-ee5bde2d0359, 6, Finished, Available, Finished, False)

gold_dim_product: 32,951 rows


## 5. Dimension: Seller

**Grain:** one row per seller.

Expected: 3,095 rows.

In [5]:
dim_seller = sellers.select(
    "seller_id",
    F.col("seller_zip_code_prefix").alias("zip_prefix"),
    F.col("seller_city").alias("city"),
    F.col("seller_state").alias("state"),
)
save_gold(dim_seller, "dim_seller")

StatementMeta(, d637f757-c75f-4c7b-ae00-ee5bde2d0359, 7, Finished, Available, Finished, False)

gold_dim_seller: 3,095 rows


## 6. Fact: Orders

**Grain:** one row per order (99,441 rows — exactly the number of Silver orders).

Items, payments and reviews are each **aggregated to order level first**, then joined one-to-one, so no join can multiply rows:
- items → `item_count`, `items_value`, `freight_value`
- payments → `payment_count`, `payment_value`, `max_installments`, `primary_payment_type` (type with the largest value)
- reviews → `review_score` of the latest review, `review_count`

Orders without items or payments (e.g. canceled orders) keep their row with zero totals.

In [6]:
item_agg = items.groupBy("order_id").agg(
    F.count("*").alias("item_count"),
    F.sum("price").alias("items_value"),
    F.sum("freight_value").alias("freight_value"))

pay_agg = payments.groupBy("order_id").agg(
    F.count("*").alias("payment_count"),
    F.sum("payment_value").alias("payment_value"),
    F.max("payment_installments").alias("max_installments"))

largest_payment = Window.partitionBy("order_id").orderBy(F.col("payment_value").desc(), F.col("payment_sequential"))
pay_primary = (payments.withColumn("_rn", F.row_number().over(largest_payment)).filter("_rn = 1")
               .select("order_id", F.col("payment_type").alias("primary_payment_type")))

latest_review = Window.partitionBy("order_id").orderBy(
    F.col("review_creation_date").desc(), F.col("review_answer_timestamp").desc(), F.col("review_id").desc())
review_agg = (reviews.withColumn("_rn", F.row_number().over(latest_review)).filter("_rn = 1")
              .select("order_id", "review_score")
              .join(reviews.groupBy("order_id").agg(F.count("*").alias("review_count")), "order_id"))

delivered = (F.col("order_status") == "delivered") & F.col("order_delivered_customer_date").isNotNull()
delay_days = F.datediff(F.col("order_delivered_customer_date"), F.col("order_estimated_delivery_date"))

fact_orders = (
    orders
    .join(cust.select("customer_id", "customer_unique_id", "customer_state"), "customer_id", "left")
    .join(item_agg, "order_id", "left").join(pay_agg, "order_id", "left")
    .join(pay_primary, "order_id", "left").join(review_agg, "order_id", "left")
    .select(
        "order_id", "customer_unique_id", "customer_state", "order_status",
        F.date_format("order_purchase_timestamp", "yyyyMMdd").cast("int").alias("order_date_key"),
        F.to_date("order_purchase_timestamp").alias("order_date"),
        "order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date",
        (~F.col("order_status").isin("canceled", "unavailable")).alias("is_revenue_order"),
        delivered.alias("is_delivered"),
        F.when(delivered, F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("delivery_days"),
        F.when(delivered, delay_days).alias("delivery_delay_days"),
        F.coalesce(delivered & (delay_days > 0), F.lit(False)).alias("is_late"),
        F.coalesce("item_count", F.lit(0)).alias("item_count"),
        F.coalesce("items_value", F.lit(0)).alias("items_value"),
        F.coalesce("freight_value", F.lit(0)).alias("freight_value"),
        F.coalesce("payment_count", F.lit(0)).alias("payment_count"),
        F.coalesce("payment_value", F.lit(0)).alias("payment_value"),
        "max_installments", "primary_payment_type",
        "review_score", F.coalesce("review_count", F.lit(0)).alias("review_count"),
    )
)
save_gold(fact_orders, "fact_orders")

StatementMeta(, d637f757-c75f-4c7b-ae00-ee5bde2d0359, 8, Finished, Available, Finished, False)

gold_fact_orders: 99,441 rows


## 7. Fact: Order Items

**Grain:** one row per order item (112,650 rows — exactly the number of Silver items).

Each item receives only **order-level attributes** (customer, date, status, delivery flags, review score) through a one-to-one join on `order_id`. Payment rows are never joined here.

In [7]:
fact_order_items = (
    items.join(
        spark.table("gold_fact_orders").select(
            "order_id", "customer_unique_id", "order_date_key", "order_date", "order_status",
            "is_revenue_order", "is_delivered", "is_late", "review_score"),
        "order_id", "inner")
    .select(
        "order_id", "order_item_id", "customer_unique_id", "product_id", "seller_id",
        "order_date_key", "order_date", "order_status", "is_revenue_order",
        "price", "freight_value", (F.col("price") + F.col("freight_value")).alias("item_total"),
        "is_delivered", "is_late", "review_score",
    )
)
save_gold(fact_order_items, "fact_order_items")

StatementMeta(, d637f757-c75f-4c7b-ae00-ee5bde2d0359, 9, Finished, Available, Finished, False)

gold_fact_order_items: 112,650 rows


## 8. Analytics: Daily Sales

**Grain:** one row per purchase day. Revenue orders only. Built from `gold_fact_order_items`.

Measures: orders, items sold, GMV, freight, distinct customers, average order value (GMV / orders). Orders that have no items (no value) do not appear.

In [8]:
revenue_items = spark.table("gold_fact_order_items").filter("is_revenue_order")

daily_sales = (
    revenue_items.groupBy(F.col("order_date_key").alias("date_key"), "order_date")
    .agg(F.countDistinct("order_id").alias("orders"),
         F.count("*").alias("items_sold"),
         F.sum("price").alias("gmv"),
         F.sum("freight_value").alias("freight"),
         F.countDistinct("customer_unique_id").alias("customers"))
    .withColumn("avg_order_value", F.round(F.col("gmv") / F.col("orders"), 2))
)
save_gold(daily_sales, "daily_sales")

StatementMeta(, d637f757-c75f-4c7b-ae00-ee5bde2d0359, 10, Finished, Available, Finished, False)

gold_daily_sales: 614 rows


## 9. Analytics: Payment Method Performance

**Grain:** one row per payment type. Revenue orders only. Built from Silver payments joined one-to-one to `gold_fact_orders` (no items involved).

Measures: orders, payment rows, total payment value, share of total value, average instalments, average payment value.

In [9]:
payment_method_performance = (
    payments.join(spark.table("gold_fact_orders").select("order_id", "is_revenue_order"), "order_id")
    .filter("is_revenue_order")
    .groupBy("payment_type")
    .agg(F.countDistinct("order_id").alias("orders"),
         F.count("*").alias("payment_rows"),
         F.sum("payment_value").alias("payment_value"),
         F.round(F.avg("payment_installments"), 2).alias("avg_installments"),
         F.round(F.avg("payment_value"), 2).alias("avg_payment_value"))
    .withColumn("share_of_payment_value",
                F.round(F.col("payment_value") / F.sum("payment_value").over(Window.partitionBy()), 4))
)
save_gold(payment_method_performance, "payment_method_performance")

StatementMeta(, d637f757-c75f-4c7b-ae00-ee5bde2d0359, 11, Finished, Available, Finished, False)

gold_payment_method_performance: 4 rows


## 10. Analytics: Product Category Sales

**Grain:** one row per product category. Revenue orders only. Built from `gold_fact_order_items` and `gold_dim_product` (one product row per product, so the join cannot multiply rows).

Review score and late rate are **order-level** values attributed to each item, so they are item-weighted averages.

In [10]:
product_category_sales = (
    revenue_items.join(spark.table("gold_dim_product").select("product_id", "category"), "product_id", "left")
    .withColumn("category", F.coalesce("category", F.lit("unknown")))
    .groupBy("category")
    .agg(F.count("*").alias("units_sold"),
         F.countDistinct("order_id").alias("orders"),
         F.sum("price").alias("revenue"),
         F.round(F.avg("price"), 2).alias("avg_price"),
         F.round(F.avg("review_score"), 2).alias("avg_review_score"),
         F.round(F.avg(F.when(F.col("is_delivered"), F.col("is_late").cast("int"))), 4).alias("late_delivery_rate"))
)
save_gold(product_category_sales, "product_category_sales")

StatementMeta(, d637f757-c75f-4c7b-ae00-ee5bde2d0359, 12, Finished, Available, Finished, False)

gold_product_category_sales: 74 rows


## 11. Analytics: Customer Summary

**Grain:** one row per real customer (`customer_unique_id`) with at least one revenue order that has items.

Measures: orders, total spent, first and last order date, recency (days from the last order date in the data to the customer's last order), repeat-customer flag, average review score.

In [11]:
snapshot_date = spark.table("gold_fact_orders").agg(F.max("order_date")).first()[0]

customer_summary = (
    revenue_items.groupBy("customer_unique_id")
    .agg(F.countDistinct("order_id").alias("orders"),
         F.sum("price").alias("total_spent"),
         F.min("order_date").alias("first_order_date"),
         F.max("order_date").alias("last_order_date"))
    .join(spark.table("gold_fact_orders").filter("is_revenue_order").groupBy("customer_unique_id")
          .agg(F.round(F.avg("review_score"), 2).alias("avg_review_score")),
          "customer_unique_id", "left")
    .withColumn("avg_order_value", F.round(F.col("total_spent") / F.col("orders"), 2))
    .withColumn("recency_days", F.datediff(F.lit(snapshot_date), F.col("last_order_date")))
    .withColumn("is_repeat_customer", F.col("orders") > 1)
)
save_gold(customer_summary, "customer_summary")

StatementMeta(, d637f757-c75f-4c7b-ae00-ee5bde2d0359, 13, Finished, Available, Finished, False)

gold_customer_summary: 94,983 rows


## 12. Analytics: Delivery Performance

**Grain:** one row per customer state and purchase month. Delivered orders only. Built from `gold_fact_orders` (state is the customer's state **at the time of the order**).

Measures: delivered orders, average delivery days, late orders, on-time rate, average delay in days (negative = early).

In [12]:
delivery_performance = (
    spark.table("gold_fact_orders").filter("is_delivered")
    .withColumn("order_month", F.date_format("order_date", "yyyy-MM"))
    .groupBy(F.col("customer_state").alias("state"), "order_month")
    .agg(F.count("*").alias("delivered_orders"),
         F.round(F.avg("delivery_days"), 2).alias("avg_delivery_days"),
         F.sum(F.col("is_late").cast("int")).alias("late_orders"),
         F.round(F.avg("delivery_delay_days"), 2).alias("avg_delay_days"))
    .withColumn("on_time_rate", F.round(1 - F.col("late_orders") / F.col("delivered_orders"), 4))
)
save_gold(delivery_performance, "delivery_performance")

StatementMeta(, d637f757-c75f-4c7b-ae00-ee5bde2d0359, 14, Finished, Available, Finished, False)

gold_delivery_performance: 556 rows


## 13. Validate the Gold Layer

Each check compares a Gold value with an independent reference (Silver tables or another Gold table). Any `FAIL` stops the notebook.

- Fact tables must have exactly the grain row counts of their Silver sources, with unique keys (proves no row multiplication or loss).
- Totals in the analytics tables must equal the totals in the fact tables.

In [13]:
fo = spark.table("gold_fact_orders")
fi = spark.table("gold_fact_order_items")
total = lambda df, c: df.agg(F.sum(c)).first()[0]
rows = []

def check(name, actual, expected):
    rows.append((name, str(actual), str(expected), "OK" if actual == expected else "FAIL"))

check("fact_orders rows = silver_orders", fo.count(), orders.count())
check("fact_orders unique order_id", fo.select("order_id").distinct().count(), fo.count())
check("fact_orders orders without a customer", fo.filter("customer_unique_id IS NULL").count(), 0)
check("fact_order_items rows = silver_order_items", fi.count(), items.count())
check("fact_order_items unique (order_id, order_item_id)", fi.select("order_id", "order_item_id").distinct().count(), fi.count())
check("fact_order_items total price = silver total price", total(fi, "price"), total(items, "price"))
check("fact_orders items_value = silver total price", total(fo, "items_value"), total(items, "price"))
check("fact_orders payment_value = silver total payments", total(fo, "payment_value"), total(payments, "payment_value"))
check("dim_customer rows = distinct customer_unique_id", spark.table("gold_dim_customer").count(), cust.select("customer_unique_id").distinct().count())
check("dim_product rows = silver_products", spark.table("gold_dim_product").count(), products.count())
check("dim_seller rows = silver_sellers", spark.table("gold_dim_seller").count(), sellers.count())
dim_date_df = spark.table("gold_dim_date")   # one DataFrame reference, so the join condition can use its columns
check("orders with a date missing in dim_date", fo.join(dim_date_df, fo["order_date_key"] == dim_date_df["date_key"], "left_anti").count(), 0)
rev_fi = fi.filter("is_revenue_order")
check("daily_sales GMV = fact GMV", total(spark.table("gold_daily_sales"), "gmv"), total(rev_fi, "price"))
check("daily_sales orders = distinct revenue orders with items", total(spark.table("gold_daily_sales"), "orders"), rev_fi.select("order_id").distinct().count())
check("category sales revenue = fact GMV", total(spark.table("gold_product_category_sales"), "revenue"), total(rev_fi, "price"))
check("customer_summary spent = fact GMV", total(spark.table("gold_customer_summary"), "total_spent"), total(rev_fi, "price"))
check("customer_summary orders = distinct revenue orders with items", total(spark.table("gold_customer_summary"), "orders"), rev_fi.select("order_id").distinct().count())
check("payment mart value = payments of revenue orders", total(spark.table("gold_payment_method_performance"), "payment_value"), total(fo.filter("is_revenue_order"), "payment_value"))
check("delivery_performance orders = delivered orders", total(spark.table("gold_delivery_performance"), "delivered_orders"), fo.filter("is_delivered").count())

report = spark.createDataFrame(rows, ["check", "actual", "expected", "status"])
display(report)
assert report.filter("status = 'FAIL'").count() == 0, "Gold validation failed - inspect the FAIL rows; do not change expected values."
print(f"Gold layer validated: {len(rows)} checks passed.")

StatementMeta(, d637f757-c75f-4c7b-ae00-ee5bde2d0359, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4d7b719b-268b-47d1-a955-bfafd80b91fd)

Gold layer validated: 19 checks passed.


## 14. Data Observations (information, not pass/fail)

These are properties of the source data that Gold deliberately keeps visible. They are reported, not forced to any value.

- Orders whose payments differ from items + freight by more than 1.00 (vouchers, instalment rounding in the source).
- Orders marked `delivered` without a delivery date.
- Orders with more than one review.
- Products whose category has no English translation, and products without any category.

In [14]:
payment_gap = fo.filter("item_count > 0 AND payment_count > 0").filter(
    F.abs(F.col("payment_value") - (F.col("items_value") + F.col("freight_value"))) > 1).count()

obs = [
    ("orders where |payments - (items + freight)| > 1.00", payment_gap),
    ("orders marked delivered without a delivery date", orders.filter("order_status = 'delivered' AND order_delivered_customer_date IS NULL").count()),
    ("orders with more than one review", fo.filter("review_count > 1").count()),
    ("revenue orders without items", fo.filter("is_revenue_order AND item_count = 0").count()),
    ("products whose category has no English translation", products.filter("product_category_name IS NOT NULL").join(cat_map, "product_category_name", "left_anti").count()),
    ("products with unknown category (no category in source)", spark.table("gold_dim_product").filter("category = 'unknown'").count()),
]
display(spark.createDataFrame(obs, ["observation", "count"]))

StatementMeta(, d637f757-c75f-4c7b-ae00-ee5bde2d0359, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 684f600f-a14e-4a3e-842b-c977fd6faac9)

## 15. Summary

| Gold table | Grain | Built from |
|---|---|---|
| `gold_dim_date` | day | Silver orders (date range) |
| `gold_dim_customer` | customer_unique_id | Silver orders + customers |
| `gold_dim_product` | product | Silver products + category translation |
| `gold_dim_seller` | seller | Silver sellers |
| `gold_fact_orders` | order | Silver orders + order-level aggregates of items, payments, reviews |
| `gold_fact_order_items` | order item | Silver items + order attributes |
| `gold_daily_sales` | day | fact_order_items |
| `gold_payment_method_performance` | payment type | Silver payments + fact_orders |
| `gold_product_category_sales` | category | fact_order_items + dim_product |
| `gold_customer_summary` | customer_unique_id | fact_order_items + fact_orders |
| `gold_delivery_performance` | state × month | fact_orders |

**Limits to state honestly:** Gold is rebuilt in full on every run (no incremental processing); average review score and late rate in the category table are item-weighted; the source is a simulated delivery of a static public dataset.

Next: SQL analytics queries and the Power BI report on top of these tables.
